Preliminary clean-up: finding duplicates from spreadsheets downloaded from Covidence and extraction spreadsheets from Drive

In [5]:
""" 
Import packages
"""
from pathlib import Path #for paths
#import pathlib
#import os
import math #to check if something is na
import numpy as np
import re #regular expressions
import pandas as pd #data frames

In [6]:
"""
This function takes in an input dataframe (IpDf) and a string for the column with paper titles (TitleColStr), extracts the title column, processes it such that the title is 
converted to lower case, all spaces and non-alphanumeric characters are removed, and then, in a second pass, all numbers are also removed. This is to find duplicate titles and 
these steps make it more likely to be able to flag duplicate titles. The PrintOrNo input controls whether the duplicate titles (based on Title_ProcessedLwr_dupes and 
Title_ProcessedLwrNoNums_dupes; see below) are printed to console

Outputs: Title_ProcessedLwr: processed title column (a series object), lower case, all non-alphanumeric characters removed
         Title_ProcessedLwr_dupes: duplicate titles (a series object) based on Title_ProcessedLwr
         Title_ProcessedLwrNoNums: processed title column with additionally all numbers also removed
         Title_ProcessedLwrNoNums_dupes: duplicate titles based on Title_ProcessedLwrNoNums

"""
def FindUniqTitles(IpDf,TitleColStr,PrintOrNo):
    TitleCol = IpDf[TitleColStr] #get the title column. This is now a series object whose labels are row indices based on IpDf. That is, the 3rd element in IpDf will be labeled 
    # by 2 (starting at 0 index). As such, this can be indexed either using IpDf's row indices as labels (using .loc()) or using the numerical indices in TitleCol (using .iloc()). 
    # Because we aren't doing any filtering at this stage, row indices for IpDf are the same as the numerical indices (i.e, the integer position) for TitleCol. 
    Title_ProcessedLwr = TitleCol.astype(str).str.lower().str.replace(r'[^a-z0-9]+','',regex=True) #converts to lower case and processes out all non-alphanumeric characters. 
    # - astype(str) handles NaN or numerical values (expectation is that titles are strings)
    # - The '+' is an optimisation choice by replacing one or more (as opposed to *, which replaces 0 or more) instances of non-alphanumeric characters, and thus, will replace a 
    #   block in one go (e.g., ' ./'). 
    #       - '*' instead of '+' here would work because of empty string replacement but for any other replacement character (say, a hyphen), all zero-length space characters would 
    #          be replaced by a hyphen. e.g., 'abc' would be processed into '-a-b-c-'
    # - pandas' astype() is being used here, but lower() needs to be accessed using pandas.Series.str (ergo the str.lower()). Also uses pandas.Series.str.replace() vs. 
    #   pandas.Series.replace() (the latter can be used by removing the 'str' in 'str.replace()'). 
    Title_ProcessedLwrNoNums = TitleCol.astype(str).str.lower().str.replace(r'[^a-z]','',regex=True) #additionally removes numeric characters

    #find duplicated titles (both for Title_ProcessedLwr and Title_ProcessedLwrNoNums) 
    Title_ProcessedLwr_dupes = Title_ProcessedLwr[Title_ProcessedLwr.duplicated()]
    Title_ProcessedLwrNoNums_dupes = Title_ProcessedLwrNoNums[Title_ProcessedLwrNoNums.duplicated()]

    if PrintOrNo: #optional print statement
        print(Title_ProcessedLwr_dupes)
        print(Title_ProcessedLwrNoNums_dupes)

    return(Title_ProcessedLwr,Title_ProcessedLwr_dupes,Title_ProcessedLwrNoNums,Title_ProcessedLwrNoNums_dupes)



In [7]:
def BasicChecksCovCsv_vs_ExtracSpreadsheets(SpreadSheetsPath,CovidenceSheetName,PreOrPostCorrec,PreCorrecExtractSheetName):
    #Covidence .csv file: contains info about all papers (pre- and post-hoc) included in the systematic search (after excluding papers based on title, 
    # abstract, and full text review as applicable) on Covidence. 
    if PreOrPostCorrec == 'Pre':
        Included_CovidenceSheet = pd.read_csv(SpreadSheetsPath/CovidenceSheetName)
    elif PreOrPostCorrec == 'Post':
        Included_CovidenceSheet = pd.read_excel(SpreadSheetsPath/CovidenceSheetName)

    #Spreadsheets used for data extraction: the prehoc sheet is the initial set (after excluding papers based on title, abstract, and full text review 
    # as applicable) and the posthoc sheet is the set of additional papers included based on DARCLE member suggestions (for papers and additional keywords)
    # and other manual interventions. For instance, some papers were included in posthoc after re-typing them as 'Observational' or 'Experimental'
    # papers that needed to be extracted. Note that these sheets ostensibly contain all papers in the Included_CovidenceSheet spreadsheet, but we
    #are only extracting 'Observational' and 'Experimental' papers
    PreHocExtractionSheet = pd.read_excel(SpreadSheetsPath/'DARCLE_extraction_2.16.26.xlsx',sheet_name=PreCorrecExtractSheetName)
    PostHocExtractionSheet = pd.read_excel(SpreadSheetsPath/'Post-hoc Paper Extractions_3.23.26.xlsx',sheet_name=PreCorrecExtractSheetName)
    CombinedExtractionSheet = pd.concat([PreHocExtractionSheet,PostHocExtractionSheet],axis=0) #combine both vertically
    #print(CombinedExtractionSheet)

    print(f'Number of papers in the Covidence .csv: {len(Included_CovidenceSheet['Title'])}')
    print(f'Number of papers in the combined extraction spreadsheet: {len(CombinedExtractionSheet['Title'])}\n') #These numbers should be the same

    #---1. Get processed titles + title duplicates for Covidence csv and the combined extraction spreadsheet 
    #(See function FindUniqTitles in this script).
    CovTitle_ProcLwr,CovTitle_ProcLwr_dupes,CovTitle_ProcLwrNoNums,CovTitle_ProcLwrNoNums_dupes = FindUniqTitles(Included_CovidenceSheet,
                                                                                                                        'Title',False)
    CombExtracTitle_ProcLwr,CombExtracTitle_ProcLwr_dupes,CombExtracTitle_ProcLwrNoNums,CombExtracTitle_ProcLwrNoNums_dupes \
                                                                                = FindUniqTitles(CombinedExtractionSheet,'Title',False)

    print(f'Duplicate titles in the Covidence csv:\n{CovTitle_ProcLwr_dupes}\n')
    print(f'Duplicate titles in the combined extraction spreadsheet:\n{CombExtracTitle_ProcLwr_dupes}\n')

    #---2. Check for titles that are in the combined extraction sheet but not in the covidence csv and vice versa
    TitlesInCombExtractSheetButNotCovCsv = CombExtracTitle_ProcLwr[~CombExtracTitle_ProcLwr.isin(CovTitle_ProcLwr)]
    TitlesInCovCsvButNotCombExtractSheet = CovTitle_ProcLwr[~CovTitle_ProcLwr.isin(CombExtracTitle_ProcLwr)]

    print(f'Titles in the combined extraction sheet but not in the Covidence csv:\n{TitlesInCombExtractSheetButNotCovCsv}\n')
    print(f'Titles in the Covidence csv but not in the combined extraction sheet:\n{TitlesInCovCsvButNotCombExtractSheet}\n')

    #---3. DOI cuplicate check (only the Covidence csv has DOIs)
    Included_CovidenceSheet_Doi = Included_CovidenceSheet['DOI']
    CovidenceDOI_dupes = Included_CovidenceSheet_Doi[Included_CovidenceSheet_Doi.duplicated()]
    CovidenceDOI_dupes_NanRem = CovidenceDOI_dupes.dropna() #if there are no DOIs, that shows up as NaN. Remove those
    CovidenceSheetDoiDupedRows = Included_CovidenceSheet[Included_CovidenceSheet['DOI'].isin(CovidenceDOI_dupes_NanRem)]

    print(f'Titles in the Covidence csv that have the same DOIs:\n{CovidenceSheetDoiDupedRows[['Title','DOI','Authors']]}\n')

In [8]:
""" 
Paths + file read-in
"""
MainPath = Path('~/Desktop/GoogleDriveFiles/research/DARCLEPaper2025/').expanduser()
SpreadSheetsPath = MainPath/'DARCLEPaper2025_ExtractionCode'/'DataToProcess_PostAutoAndManualExtraction_2026_04'
#print(SpreadSheetsPath)

PreOrPostCorrec = 'Pre'
PreCorrecCovCsvName = 'CovidenceFinalExport_2026_05.csv'
PreCorrecExtractSheetName = 'ReTyped_PostManualExt'
print(f'These checks are being done on spreadsheets/csvs after automatic and manual extraction but BEFORE corretions based on this script\n')
BasicChecksCovCsv_vs_ExtracSpreadsheets(SpreadSheetsPath,PreCorrecCovCsvName,PreOrPostCorrec,PreCorrecExtractSheetName)

PreOrPostCorrec = 'Post'
PostCorrecCovCsvName = 'CovidenceFinalExport_2026_05_DupesRem.xlsx'
PostCorrecExtractSheetName = 'Corrected_ReTyped_PostManualExt'
print(f'These checks are being done on spreadsheets/csvs after automatic and manual extraction but AFTER corretions based on this script\n')
BasicChecksCovCsv_vs_ExtracSpreadsheets(SpreadSheetsPath,PostCorrecCovCsvName,PreOrPostCorrec,PostCorrecExtractSheetName)

These checks are being done on spreadsheets/csvs after automatic and manual extraction but BEFORE corretions based on this script

Number of papers in the Covidence .csv: 386
Number of papers in the combined extraction spreadsheet: 384

Duplicate titles in the Covidence csv:
331    associationsbetweenmaternalstressearlylanguage...
376    homelanguageenvironmentinrelationtolanguageout...
Name: Title, dtype: object

Duplicate titles in the combined extraction spreadsheet:
41    homelanguageenvironmentinrelationtolanguageout...
Name: Title, dtype: object

Titles in the combined extraction sheet but not in the Covidence csv:
Series([], Name: Title, dtype: object)

Titles in the Covidence csv but not in the combined extraction sheet:
377    hearttohearttheartsofinfantversusadultdirected...
Name: Title, dtype: object

Titles in the Covidence csv that have the same DOIs:
                                                 Title  \
2    The Predictability of Naturalistic Evaluation ...   
186  Re